In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# Ambush Hypothesis Analysis for 15-mers
#
# This code:
# 1. Reads CDS sequences from CSV
# 2. Uses Biopython translation tables
# 3. Checks stop codons in:
#       original frame
#       +1 shifted frame
#       +2 shifted frame
# 4. Counts mapped 15mers for +1 and +2 shifts
# 5. Finds stop codons before the first mapped 15mer
# 6. Finds stop codons inside mapped 15mers
# 7. Performs paired t-test + Bonferroni correction
# 8. Saves detailed results, summary files, and inferential statistics
# ============================================================

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install biopython statsmodels -q

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable
import re

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from itertools import combinations


# ============================================================
# FILE PATHS
# ============================================================

input_file = "/content/final_pep_frames_tranls.csv"

output_file = "/content/ambush_hypothesis_results_with_first15mer.csv"
summary_output_file = "/content/ambush_hypothesis_summary_with_first15mer.csv"

mapped15mer_output_file = "/content/mapped15mer_stopcodon_analysis.csv"
mapped15mer_summary_file = "/content/mapped15mer_stopcodon_summary.csv"

inferential_output_file = "/content/stop_count_ttest_bonferroni_results_15mer.csv"

# ============================================================
# CLEAN FUNCTIONS
# ============================================================

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = seq.replace("U", "T")
    seq = re.sub(r"[^ATGC]", "", seq)
    return seq


def normalize_shift(x):
    x = str(x).strip()

    if x in ["1", "1.0"]:
        return "1"

    if x in ["2", "2.0"]:
        return "2"

    if x in ["transl_table_issue", "trans_table_issue"]:
        return "transl_table_issue"

    return x


def normalize_transl_table(x):
    try:
        return int(float(str(x).strip()))
    except Exception:
        return None


def clean_position_plus_one(pos):
    try:
        pos = str(pos).strip()

        if "|" in pos:
            pos = pos.split("|")[0]

        return int(float(pos)) + 1

    except Exception:
        return None


def get_shifted_codons(seq, shift):
    shifted_seq = seq[shift:]
    codons = []

    for i in range(0, len(shifted_seq) - 2, 3):
        codon = shifted_seq[i:i + 3]

        if len(codon) == 3:
            codons.append(codon)

    return codons


def find_stop_codons(codons, stop_codons):
    stop_positions = []
    stop_codon_list = []

    for idx, codon in enumerate(codons):
        if codon in stop_codons:
            stop_positions.append(idx + 1)
            stop_codon_list.append(codon)

    return stop_codon_list, stop_positions


def count_stop_codons_before_nt_position(seq, frame_shift, stop_codons, nt_position):
    """
    Counts stop codons before the first mapped 15mer.

    nt_position is 1-based nucleotide position.
    frame_shift:
        0 = original frame
        1 = +1 frame
        2 = +2 frame
    """

    if nt_position is None or pd.isna(nt_position):
        return 0, "", ""

    codons = get_shifted_codons(seq, frame_shift)

    stop_count = 0
    stop_codon_list = []
    stop_position_list = []

    for idx, codon in enumerate(codons):
        codon_start_0based = frame_shift + idx * 3
        codon_start_1based = codon_start_0based + 1

        if codon_start_1based >= nt_position:
            break

        if codon in stop_codons:
            stop_count += 1
            stop_codon_list.append(codon)
            stop_position_list.append(idx + 1)

    return (
        stop_count,
        "|".join(stop_codon_list),
        "|".join(map(str, stop_position_list))
    )


def count_stop_codons_inside_15mer_all_frames(seq15, stop_codons):
    """
    Counts stop codons inside a 15mer across internal frames 0, 1, and 2.
    """

    seq15 = clean_sequence(seq15)

    stop_count = 0
    found_stop_codons = []
    found_internal_frames = []
    found_positions = []

    for frame in [0, 1, 2]:
        codons = get_shifted_codons(seq15, frame)

        for idx, codon in enumerate(codons):
            if codon in stop_codons:
                stop_count += 1
                found_stop_codons.append(codon)
                found_internal_frames.append(str(frame))
                found_positions.append(str(idx + 1))

    return (
        stop_count,
        "|".join(found_stop_codons),
        "|".join(found_internal_frames),
        "|".join(found_positions)
    )


# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(
    input_file,
    dtype=str,
    low_memory=False
)

print("\nDataset loaded.")
print("Total rows:", len(df))

# ============================================================
# CLEAN DATA
# ============================================================

print("\nCleaning dataset...")

df["shift"] = df["shift"].apply(normalize_shift)
df["cds_sequence_clean"] = df["cds_sequence"].apply(clean_sequence)
df["matched_15mer_clean"] = df["matched_15mer"].apply(clean_sequence)
df["transl_table_clean"] = df["transl_table"].apply(normalize_transl_table)
df["position_corrected_plus_one"] = df["positions"].apply(clean_position_plus_one)

print("\nShift distribution:")
print(df["shift"].value_counts(dropna=False))

print("\nTranslation table distribution:")
print(df["transl_table_clean"].value_counts(dropna=False))

# ============================================================
# BUILD UNIQUE CDS TABLE
# ============================================================

print("\nBuilding unique CDS table...")

unique_cds = df[
    ["cds_sequence_clean", "transl_table_clean"]
].drop_duplicates()

unique_cds = unique_cds.dropna(
    subset=["transl_table_clean"]
)

unique_cds["transl_table_clean"] = unique_cds[
    "transl_table_clean"
].astype(int)

print(
    "Unique CDS + translation table combinations:",
    len(unique_cds)
)

# ============================================================
# COUNT MAPPED 15MERS AND FIRST MAPPED 15MER POSITION
# ============================================================

print("\nCounting mapped 15mers and first mapped 15mer position...")

shift1_counts = defaultdict(int)
shift2_counts = defaultdict(int)
transl_table_issue_counts = defaultdict(int)
all_mapped_counts = defaultdict(int)

first_mapped_position = {}

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Counting mapped 15mers"
):

    cds = row["cds_sequence_clean"]
    shift = row["shift"]
    pos = row["position_corrected_plus_one"]

    if shift == "1":
        shift1_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "2":
        shift2_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "transl_table_issue":
        transl_table_issue_counts[cds] += 1
        all_mapped_counts[cds] += 1

    if pos is not None and not pd.isna(pos):
        if cds not in first_mapped_position:
            first_mapped_position[cds] = pos
        else:
            first_mapped_position[cds] = min(
                first_mapped_position[cds],
                pos
            )

# ============================================================
# MAIN CDS-LEVEL AMBUSH ANALYSIS
# ============================================================

print(
    "\nRunning original-frame, +1-frame, +2-frame, "
    "and before-first-15mer stop codon analysis..."
)

results = []
invalid_translation_tables = []

for _, row in tqdm(
    unique_cds.iterrows(),
    total=len(unique_cds),
    desc="CDS-level stop codon analysis"
):

    cds_seq = row["cds_sequence_clean"]
    transl_table_id = row["transl_table_clean"]

    try:
        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        invalid_translation_tables.append(transl_table_id)
        continue

    # Original frame
    original_codons = get_shifted_codons(cds_seq, 0)
    original_stop_codons, original_stop_positions = find_stop_codons(
        original_codons,
        stop_codons
    )

    # +1 frame
    plus1_codons = get_shifted_codons(cds_seq, 1)
    plus1_stop_codons, plus1_stop_positions = find_stop_codons(
        plus1_codons,
        stop_codons
    )

    # +2 frame
    plus2_codons = get_shifted_codons(cds_seq, 2)
    plus2_stop_codons, plus2_stop_positions = find_stop_codons(
        plus2_codons,
        stop_codons
    )

    # Stop codons before first mapped 15mer
    first_pos = first_mapped_position.get(cds_seq, np.nan)

    original_before_count, original_before_codons, original_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            0,
            stop_codons,
            first_pos
        )
    )

    plus1_before_count, plus1_before_codons, plus1_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            1,
            stop_codons,
            first_pos
        )
    )

    plus2_before_count, plus2_before_codons, plus2_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            2,
            stop_codons,
            first_pos
        )
    )

    results.append({

        "cds_sequence": cds_seq,
        "translation_table": transl_table_id,
        "stop_codons_for_table": "|".join(sorted(stop_codons)),
        "cds_length_nt": len(cds_seq),
        "first_mapped_15mer_nt_position": first_pos,

        # Original frame
        "original_total_codons": len(original_codons),
        "original_stop_count": len(original_stop_codons),
        "original_stop_density_per_100_codons": (
            len(original_stop_codons) / len(original_codons) * 100
        ) if len(original_codons) > 0 else np.nan,
        "original_stop_codons": "|".join(original_stop_codons),
        "original_stop_positions": "|".join(map(str, original_stop_positions)),

        # +1 frame
        "plus1_total_codons": len(plus1_codons),
        "plus1_stop_count": len(plus1_stop_codons),
        "plus1_stop_density_per_100_codons": (
            len(plus1_stop_codons) / len(plus1_codons) * 100
        ) if len(plus1_codons) > 0 else np.nan,
        "plus1_stop_codons": "|".join(plus1_stop_codons),
        "plus1_stop_positions": "|".join(map(str, plus1_stop_positions)),
        "plus1_mapped_15mers": shift1_counts.get(cds_seq, 0),

        # +2 frame
        "plus2_total_codons": len(plus2_codons),
        "plus2_stop_count": len(plus2_stop_codons),
        "plus2_stop_density_per_100_codons": (
            len(plus2_stop_codons) / len(plus2_codons) * 100
        ) if len(plus2_codons) > 0 else np.nan,
        "plus2_stop_codons": "|".join(plus2_stop_codons),
        "plus2_stop_positions": "|".join(map(str, plus2_stop_positions)),
        "plus2_mapped_15mers": shift2_counts.get(cds_seq, 0),

        # Mapped 15mer summary
        "transl_table_issue_mapped_15mers": transl_table_issue_counts.get(cds_seq, 0),

        "total_mapped_15mers_shift1_shift2": (
            shift1_counts.get(cds_seq, 0)
            +
            shift2_counts.get(cds_seq, 0)
        ),

        "total_mapped_15mers_all_tracked_groups": all_mapped_counts.get(cds_seq, 0),

        # Stop codons before first mapped 15mer
        "original_stop_count_before_first_mapped_15mer": original_before_count,
        "original_stop_codons_before_first_mapped_15mer": original_before_codons,
        "original_stop_positions_before_first_mapped_15mer": original_before_positions,

        "plus1_stop_count_before_first_mapped_15mer": plus1_before_count,
        "plus1_stop_codons_before_first_mapped_15mer": plus1_before_codons,
        "plus1_stop_positions_before_first_mapped_15mer": plus1_before_positions,

        "plus2_stop_count_before_first_mapped_15mer": plus2_before_count,
        "plus2_stop_codons_before_first_mapped_15mer": plus2_before_codons,
        "plus2_stop_positions_before_first_mapped_15mer": plus2_before_positions
    })

# ============================================================
# MAPPED 15MER-LEVEL STOP CODON ANALYSIS
# ============================================================

print("\nRunning mapped 15mer-level stop codon analysis...")

mapped15_results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Mapped 15mer stop codon analysis"
):

    transl_table_id = row["transl_table_clean"]

    if pd.isna(transl_table_id):
        continue

    try:
        transl_table_id = int(transl_table_id)

        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        continue

    seq15 = row["matched_15mer_clean"]

    stop_count, stop_codons_found, internal_frames, internal_positions = (
        count_stop_codons_inside_15mer_all_frames(
            seq15,
            stop_codons
        )
    )

    mapped15_results.append({

        "cds_sequence_clean": row["cds_sequence_clean"],
        "matched_15mer_clean": seq15,
        "shift": row["shift"],
        "translation_table": transl_table_id,

        "mapped_15mer_has_stop_codon": stop_count > 0,
        "mapped_15mer_stop_codon_count": stop_count,
        "mapped_15mer_stop_codons": stop_codons_found,
        "mapped_15mer_stop_internal_frames": internal_frames,
        "mapped_15mer_stop_internal_positions": internal_positions
    })

# ============================================================
# SAVE DETAILED OUTPUTS
# ============================================================

results_df = pd.DataFrame(results)
mapped15_df = pd.DataFrame(mapped15_results)

# ============================================================
# INFERENTIAL STATISTICS
# Paired t-test + Bonferroni correction
# Comparing original_stop_count, plus1_stop_count, plus2_stop_count
# ============================================================

stop_count_cols = [
    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count"
]

stats_df = results_df[stop_count_cols].copy()

for col in stop_count_cols:
    stats_df[col] = pd.to_numeric(stats_df[col], errors="coerce")

stats_df = stats_df.dropna()

ttest_results = []

for col1, col2 in combinations(stop_count_cols, 2):

    t_stat, p_value = ttest_rel(
        stats_df[col1],
        stats_df[col2],
        nan_policy="omit"
    )

    ttest_results.append({
        "Comparison": f"{col1} vs {col2}",
        "Variable_1": col1,
        "Variable_2": col2,
        "Mean_1": stats_df[col1].mean(),
        "SD_1": stats_df[col1].std(ddof=1),
        "Mean_2": stats_df[col2].mean(),
        "SD_2": stats_df[col2].std(ddof=1),
        "N": len(stats_df),
        "t_statistic": t_stat,
        "raw_p_value": p_value
    })

inferential_df = pd.DataFrame(ttest_results)

reject, corrected_pvals, _, _ = multipletests(
    inferential_df["raw_p_value"],
    alpha=0.05,
    method="bonferroni"
)

inferential_df["bonferroni_corrected_p"] = corrected_pvals
inferential_df["significant_after_bonferroni_95CI"] = reject

inferential_df.to_csv(
    inferential_output_file,
    index=False
)

print("\nInferential statistics saved:")
print(inferential_output_file)

print("\nStop-count paired t-test + Bonferroni results:")
print(inferential_df)

# ============================================================
# SAVE MAIN OUTPUT FILES
# ============================================================

results_df.to_csv(
    output_file,
    index=False
)

mapped15_df.to_csv(
    mapped15mer_output_file,
    index=False
)

print("\nCDS-level detailed output saved:")
print(output_file)

print("\nMapped 15mer-level output saved:")
print(mapped15mer_output_file)

print("\nTotal CDS analyzed:")
print(len(results_df))

print("\nTotal mapped 15mers analyzed:")
print(len(mapped15_df))

# ============================================================
# CDS-LEVEL SUMMARY STATISTICS
# ============================================================

summary_cols = [

    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count",

    "original_stop_density_per_100_codons",
    "plus1_stop_density_per_100_codons",
    "plus2_stop_density_per_100_codons",

    "original_stop_count_before_first_mapped_15mer",
    "plus1_stop_count_before_first_mapped_15mer",
    "plus2_stop_count_before_first_mapped_15mer"
]

summary_df = results_df[
    summary_cols
].describe()

summary_df.to_csv(
    summary_output_file
)

print("\nCDS-level summary saved:")
print(summary_output_file)

# ============================================================
# MAPPED 15MER STOP SUMMARY
# ============================================================

total_15mers = len(mapped15_df)

stop_containing_15mers = mapped15_df[
    mapped15_df["mapped_15mer_has_stop_codon"] == True
]

percent_15mers_with_stop = (
    len(stop_containing_15mers)
    / total_15mers
    * 100
    if total_15mers > 0
    else np.nan
)

average_stop_codons_all_15mers = mapped15_df[
    "mapped_15mer_stop_codon_count"
].mean()

average_stop_codons_stop_containing_only = stop_containing_15mers[
    "mapped_15mer_stop_codon_count"
].mean()

mapped15_summary = pd.DataFrame({

    "metric": [

        "total_mapped_15mers",
        "mapped_15mers_with_at_least_one_stop",
        "percent_mapped_15mers_with_stop",
        "average_stop_codons_per_15mer_all",
        "average_stop_codons_per_stop_containing_15mer_only"

    ],

    "value": [

        total_15mers,
        len(stop_containing_15mers),
        percent_15mers_with_stop,
        average_stop_codons_all_15mers,
        average_stop_codons_stop_containing_only
    ]
})

mapped15_summary.to_csv(
    mapped15mer_summary_file,
    index=False
)

print("\nMapped 15mer stop summary saved:")
print(mapped15mer_summary_file)

# ============================================================
# INVALID TRANSLATION TABLES
# ============================================================

if len(invalid_translation_tables) > 0:
    print("\nInvalid / unsupported translation table IDs encountered:")
    print(sorted(set(invalid_translation_tables)))

# ============================================================
# PREVIEW
# ============================================================

print("\nPreview of CDS-level detailed results:")
print(results_df.head())

print("\nCDS-level summary statistics:")
print(summary_df)

print("\nMapped 15mer stop summary:")
print(mapped15_summary)

print("\nInferential statistics:")
print(inferential_df)

In [ ]:
# ============================================================
# GOOGLE COLAB CODE
# Ambush Hypothesis Analysis for 12-mers
#
# This code:
# 1. Reads CDS sequences from CSV
# 2. Uses Biopython translation tables
# 3. Checks stop codons in:
#       original frame
#       +1 shifted frame
#       +2 shifted frame
# 4. Counts mapped 12mers for +1 and +2 shifts
# 5. Finds stop codons before the first mapped 12mer
# 6. Finds stop codons inside mapped 12mers
# 7. Performs paired t-test + Bonferroni correction
# 8. Saves detailed results, summary files, and inferential statistics
# ============================================================

# ============================================================
# INSTALL DEPENDENCIES
# ============================================================

!pip install biopython statsmodels -q

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import defaultdict
from Bio.Data import CodonTable
import re

from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests
from itertools import combinations


# ============================================================
# FILE PATHS
# ============================================================

input_file = "/content/CQWW_frames.csv"

output_file = "/content/ambush_hypothesis_results_with_first12mer.csv"
summary_output_file = "/content/ambush_hypothesis_summary_with_first12mer.csv"

mapped12mer_output_file = "/content/mapped12mer_stopcodon_analysis.csv"
mapped12mer_summary_file = "/content/mapped12mer_stopcodon_summary.csv"

inferential_output_file = "/content/stop_count_ttest_bonferroni_results_12mer.csv"

# ============================================================
# CLEAN FUNCTIONS
# ============================================================

def clean_sequence(seq):
    seq = str(seq).upper()
    seq = seq.replace("U", "T")
    seq = re.sub(r"[^ATGC]", "", seq)
    return seq


def normalize_shift(x):
    x = str(x).strip()

    if x in ["1", "1.0"]:
        return "1"

    if x in ["2", "2.0"]:
        return "2"

    if x in ["transl_table_issue", "trans_table_issue"]:
        return "transl_table_issue"

    return x


def normalize_transl_table(x):
    try:
        return int(float(str(x).strip()))
    except Exception:
        return None


def clean_position_plus_one(pos):
    try:
        pos = str(pos).strip()

        if "|" in pos:
            pos = pos.split("|")[0]

        return int(float(pos)) + 1

    except Exception:
        return None


def get_shifted_codons(seq, shift):
    shifted_seq = seq[shift:]
    codons = []

    for i in range(0, len(shifted_seq) - 2, 3):
        codon = shifted_seq[i:i + 3]

        if len(codon) == 3:
            codons.append(codon)

    return codons


def find_stop_codons(codons, stop_codons):
    stop_positions = []
    stop_codon_list = []

    for idx, codon in enumerate(codons):
        if codon in stop_codons:
            stop_positions.append(idx + 1)
            stop_codon_list.append(codon)

    return stop_codon_list, stop_positions


def count_stop_codons_before_nt_position(seq, frame_shift, stop_codons, nt_position):
    """
    Counts stop codons before the first mapped 12mer.

    nt_position is 1-based nucleotide position.
    frame_shift:
        0 = original frame
        1 = +1 frame
        2 = +2 frame
    """

    if nt_position is None or pd.isna(nt_position):
        return 0, "", ""

    codons = get_shifted_codons(seq, frame_shift)

    stop_count = 0
    stop_codon_list = []
    stop_position_list = []

    for idx, codon in enumerate(codons):
        codon_start_0based = frame_shift + idx * 3
        codon_start_1based = codon_start_0based + 1

        if codon_start_1based >= nt_position:
            break

        if codon in stop_codons:
            stop_count += 1
            stop_codon_list.append(codon)
            stop_position_list.append(idx + 1)

    return (
        stop_count,
        "|".join(stop_codon_list),
        "|".join(map(str, stop_position_list))
    )


def count_stop_codons_inside_12mer_all_frames(seq12, stop_codons):
    """
    Counts stop codons inside a 12mer across internal frames 0, 1, and 2.
    """

    seq12 = clean_sequence(seq12)

    stop_count = 0
    found_stop_codons = []
    found_internal_frames = []
    found_positions = []

    for frame in [0, 1, 2]:
        codons = get_shifted_codons(seq12, frame)

        for idx, codon in enumerate(codons):
            if codon in stop_codons:
                stop_count += 1
                found_stop_codons.append(codon)
                found_internal_frames.append(str(frame))
                found_positions.append(str(idx + 1))

    return (
        stop_count,
        "|".join(found_stop_codons),
        "|".join(found_internal_frames),
        "|".join(found_positions)
    )


# ============================================================
# LOAD DATA
# ============================================================

print("\nLoading dataset...")

df = pd.read_csv(
    input_file,
    dtype=str,
    low_memory=False
)

print("\nDataset loaded.")
print("Total rows:", len(df))

# ============================================================
# CLEAN DATA
# ============================================================

print("\nCleaning dataset...")

df["shift"] = df["shift"].apply(normalize_shift)
df["cds_sequence_clean"] = df["cds_sequence"].apply(clean_sequence)

# IMPORTANT: this assumes your tetrapeptide file has a column named matched_12mer
df["matched_12mer_clean"] = df["matched_12mer"].apply(clean_sequence)

df["transl_table_clean"] = df["transl_table"].apply(normalize_transl_table)
df["position_corrected_plus_one"] = df["positions"].apply(clean_position_plus_one)

print("\nShift distribution:")
print(df["shift"].value_counts(dropna=False))

print("\nTranslation table distribution:")
print(df["transl_table_clean"].value_counts(dropna=False))

# ============================================================
# BUILD UNIQUE CDS TABLE
# ============================================================

print("\nBuilding unique CDS table...")

unique_cds = df[
    ["cds_sequence_clean", "transl_table_clean"]
].drop_duplicates()

unique_cds = unique_cds.dropna(
    subset=["transl_table_clean"]
)

unique_cds["transl_table_clean"] = unique_cds[
    "transl_table_clean"
].astype(int)

print(
    "Unique CDS + translation table combinations:",
    len(unique_cds)
)

# ============================================================
# COUNT MAPPED 12MERS AND FIRST MAPPED 12MER POSITION
# ============================================================

print("\nCounting mapped 12mers and first mapped 12mer position...")

shift1_counts = defaultdict(int)
shift2_counts = defaultdict(int)
transl_table_issue_counts = defaultdict(int)
all_mapped_counts = defaultdict(int)

first_mapped_position = {}

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Counting mapped 12mers"
):

    cds = row["cds_sequence_clean"]
    shift = row["shift"]
    pos = row["position_corrected_plus_one"]

    if shift == "1":
        shift1_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "2":
        shift2_counts[cds] += 1
        all_mapped_counts[cds] += 1

    elif shift == "transl_table_issue":
        transl_table_issue_counts[cds] += 1
        all_mapped_counts[cds] += 1

    if pos is not None and not pd.isna(pos):
        if cds not in first_mapped_position:
            first_mapped_position[cds] = pos
        else:
            first_mapped_position[cds] = min(
                first_mapped_position[cds],
                pos
            )

# ============================================================
# MAIN CDS-LEVEL AMBUSH ANALYSIS
# ============================================================

print(
    "\nRunning original-frame, +1-frame, +2-frame, "
    "and before-first-12mer stop codon analysis..."
)

results = []
invalid_translation_tables = []

for _, row in tqdm(
    unique_cds.iterrows(),
    total=len(unique_cds),
    desc="CDS-level stop codon analysis"
):

    cds_seq = row["cds_sequence_clean"]
    transl_table_id = row["transl_table_clean"]

    try:
        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        invalid_translation_tables.append(transl_table_id)
        continue

    # Original frame
    original_codons = get_shifted_codons(cds_seq, 0)
    original_stop_codons, original_stop_positions = find_stop_codons(
        original_codons,
        stop_codons
    )

    # +1 frame
    plus1_codons = get_shifted_codons(cds_seq, 1)
    plus1_stop_codons, plus1_stop_positions = find_stop_codons(
        plus1_codons,
        stop_codons
    )

    # +2 frame
    plus2_codons = get_shifted_codons(cds_seq, 2)
    plus2_stop_codons, plus2_stop_positions = find_stop_codons(
        plus2_codons,
        stop_codons
    )

    # Stop codons before first mapped 12mer
    first_pos = first_mapped_position.get(cds_seq, np.nan)

    original_before_count, original_before_codons, original_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            0,
            stop_codons,
            first_pos
        )
    )

    plus1_before_count, plus1_before_codons, plus1_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            1,
            stop_codons,
            first_pos
        )
    )

    plus2_before_count, plus2_before_codons, plus2_before_positions = (
        count_stop_codons_before_nt_position(
            cds_seq,
            2,
            stop_codons,
            first_pos
        )
    )

    results.append({

        "cds_sequence": cds_seq,
        "translation_table": transl_table_id,
        "stop_codons_for_table": "|".join(sorted(stop_codons)),
        "cds_length_nt": len(cds_seq),
        "first_mapped_12mer_nt_position": first_pos,

        # Original frame
        "original_total_codons": len(original_codons),
        "original_stop_count": len(original_stop_codons),
        "original_stop_density_per_100_codons": (
            len(original_stop_codons) / len(original_codons) * 100
        ) if len(original_codons) > 0 else np.nan,
        "original_stop_codons": "|".join(original_stop_codons),
        "original_stop_positions": "|".join(map(str, original_stop_positions)),

        # +1 frame
        "plus1_total_codons": len(plus1_codons),
        "plus1_stop_count": len(plus1_stop_codons),
        "plus1_stop_density_per_100_codons": (
            len(plus1_stop_codons) / len(plus1_codons) * 100
        ) if len(plus1_codons) > 0 else np.nan,
        "plus1_stop_codons": "|".join(plus1_stop_codons),
        "plus1_stop_positions": "|".join(map(str, plus1_stop_positions)),
        "plus1_mapped_12mers": shift1_counts.get(cds_seq, 0),

        # +2 frame
        "plus2_total_codons": len(plus2_codons),
        "plus2_stop_count": len(plus2_stop_codons),
        "plus2_stop_density_per_100_codons": (
            len(plus2_stop_codons) / len(plus2_codons) * 100
        ) if len(plus2_codons) > 0 else np.nan,
        "plus2_stop_codons": "|".join(plus2_stop_codons),
        "plus2_stop_positions": "|".join(map(str, plus2_stop_positions)),
        "plus2_mapped_12mers": shift2_counts.get(cds_seq, 0),

        # Mapped 12mer summary
        "transl_table_issue_mapped_12mers": transl_table_issue_counts.get(cds_seq, 0),

        "total_mapped_12mers_shift1_shift2": (
            shift1_counts.get(cds_seq, 0)
            +
            shift2_counts.get(cds_seq, 0)
        ),

        "total_mapped_12mers_all_tracked_groups": all_mapped_counts.get(cds_seq, 0),

        # Stop codons before first mapped 12mer
        "original_stop_count_before_first_mapped_12mer": original_before_count,
        "original_stop_codons_before_first_mapped_12mer": original_before_codons,
        "original_stop_positions_before_first_mapped_12mer": original_before_positions,

        "plus1_stop_count_before_first_mapped_12mer": plus1_before_count,
        "plus1_stop_codons_before_first_mapped_12mer": plus1_before_codons,
        "plus1_stop_positions_before_first_mapped_12mer": plus1_before_positions,

        "plus2_stop_count_before_first_mapped_12mer": plus2_before_count,
        "plus2_stop_codons_before_first_mapped_12mer": plus2_before_codons,
        "plus2_stop_positions_before_first_mapped_12mer": plus2_before_positions
    })

# ============================================================
# MAPPED 12MER-LEVEL STOP CODON ANALYSIS
# ============================================================

print("\nRunning mapped 12mer-level stop codon analysis...")

mapped12_results = []

for _, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Mapped 12mer stop codon analysis"
):

    transl_table_id = row["transl_table_clean"]

    if pd.isna(transl_table_id):
        continue

    try:
        transl_table_id = int(transl_table_id)

        codon_table = CodonTable.unambiguous_dna_by_id[
            transl_table_id
        ]

        stop_codons = set(codon_table.stop_codons)

    except Exception:
        continue

    seq12 = row["matched_12mer_clean"]

    stop_count, stop_codons_found, internal_frames, internal_positions = (
        count_stop_codons_inside_12mer_all_frames(
            seq12,
            stop_codons
        )
    )

    mapped12_results.append({

        "cds_sequence_clean": row["cds_sequence_clean"],
        "matched_12mer_clean": seq12,
        "shift": row["shift"],
        "translation_table": transl_table_id,

        "mapped_12mer_has_stop_codon": stop_count > 0,
        "mapped_12mer_stop_codon_count": stop_count,
        "mapped_12mer_stop_codons": stop_codons_found,
        "mapped_12mer_stop_internal_frames": internal_frames,
        "mapped_12mer_stop_internal_positions": internal_positions
    })

# ============================================================
# SAVE DETAILED OUTPUTS
# ============================================================

results_df = pd.DataFrame(results)
mapped12_df = pd.DataFrame(mapped12_results)

# ============================================================
# INFERENTIAL STATISTICS
# Paired t-test + Bonferroni correction
# Comparing original_stop_count, plus1_stop_count, plus2_stop_count
# ============================================================

stop_count_cols = [
    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count"
]

stats_df = results_df[stop_count_cols].copy()

for col in stop_count_cols:
    stats_df[col] = pd.to_numeric(stats_df[col], errors="coerce")

stats_df = stats_df.dropna()

ttest_results = []

for col1, col2 in combinations(stop_count_cols, 2):

    t_stat, p_value = ttest_rel(
        stats_df[col1],
        stats_df[col2],
        nan_policy="omit"
    )

    ttest_results.append({
        "Comparison": f"{col1} vs {col2}",
        "Variable_1": col1,
        "Variable_2": col2,
        "Mean_1": stats_df[col1].mean(),
        "SD_1": stats_df[col1].std(ddof=1),
        "Mean_2": stats_df[col2].mean(),
        "SD_2": stats_df[col2].std(ddof=1),
        "N": len(stats_df),
        "t_statistic": t_stat,
        "raw_p_value": p_value
    })

inferential_df = pd.DataFrame(ttest_results)

reject, corrected_pvals, _, _ = multipletests(
    inferential_df["raw_p_value"],
    alpha=0.05,
    method="bonferroni"
)

inferential_df["bonferroni_corrected_p"] = corrected_pvals
inferential_df["significant_after_bonferroni_95CI"] = reject

inferential_df.to_csv(
    inferential_output_file,
    index=False
)

print("\nInferential statistics saved:")
print(inferential_output_file)

print("\nStop-count paired t-test + Bonferroni results:")
print(inferential_df)

# ============================================================
# SAVE MAIN OUTPUT FILES
# ============================================================

results_df.to_csv(
    output_file,
    index=False
)

mapped12_df.to_csv(
    mapped12mer_output_file,
    index=False
)

print("\nCDS-level detailed output saved:")
print(output_file)

print("\nMapped 12mer-level output saved:")
print(mapped12mer_output_file)

print("\nTotal CDS analyzed:")
print(len(results_df))

print("\nTotal mapped 12mers analyzed:")
print(len(mapped12_df))

# ============================================================
# CDS-LEVEL SUMMARY STATISTICS
# ============================================================

summary_cols = [

    "original_stop_count",
    "plus1_stop_count",
    "plus2_stop_count",

    "original_stop_density_per_100_codons",
    "plus1_stop_density_per_100_codons",
    "plus2_stop_density_per_100_codons",

    "original_stop_count_before_first_mapped_12mer",
    "plus1_stop_count_before_first_mapped_12mer",
    "plus2_stop_count_before_first_mapped_12mer"
]

summary_df = results_df[
    summary_cols
].describe()

summary_df.to_csv(
    summary_output_file
)

print("\nCDS-level summary saved:")
print(summary_output_file)

# ============================================================
# MAPPED 12MER STOP SUMMARY
# ============================================================

total_12mers = len(mapped12_df)

stop_containing_12mers = mapped12_df[
    mapped12_df["mapped_12mer_has_stop_codon"] == True
]

percent_12mers_with_stop = (
    len(stop_containing_12mers)
    / total_12mers
    * 100
    if total_12mers > 0
    else np.nan
)

average_stop_codons_all_12mers = mapped12_df[
    "mapped_12mer_stop_codon_count"
].mean()

average_stop_codons_stop_containing_only = stop_containing_12mers[
    "mapped_12mer_stop_codon_count"
].mean()

mapped12_summary = pd.DataFrame({

    "metric": [

        "total_mapped_12mers",
        "mapped_12mers_with_at_least_one_stop",
        "percent_mapped_12mers_with_stop",
        "average_stop_codons_per_12mer_all",
        "average_stop_codons_per_stop_containing_12mer_only"
    ],

    "value": [

        total_12mers,
        len(stop_containing_12mers),
        percent_12mers_with_stop,
        average_stop_codons_all_12mers,
        average_stop_codons_stop_containing_only
    ]
})

mapped12_summary.to_csv(
    mapped12mer_summary_file,
    index=False
)

print("\nMapped 12mer stop summary saved:")
print(mapped12mer_summary_file)

# ============================================================
# INVALID TRANSLATION TABLES
# ============================================================

if len(invalid_translation_tables) > 0:
    print("\nInvalid / unsupported translation table IDs encountered:")
    print(sorted(set(invalid_translation_tables)))

# ============================================================
# PREVIEW
# ============================================================

print("\nPreview of CDS-level detailed results:")
print(results_df.head())

print("\nCDS-level summary statistics:")
print(summary_df)

print("\nMapped 12mer stop summary:")
print(mapped12_summary)

print("\nInferential statistics:")
print(inferential_df)

In [ ]:
### Reverse peptide analysis

In [ ]:
import os
import pandas as pd
from collections import defaultdict, Counter

import matplotlib.pyplot as plt


def analyze_complete_peptides(file_path, out_dir="/content/"):
    print(f"Loading data from {file_path}...")

    try:
        df = pd.read_csv(file_path, usecols=["peptide", "matched_15mer", "transl_table"], low_memory=False)
    except FileNotFoundError:
        print(f"Error: '{file_path}' not found.")
        return
    except ValueError as e:
        print(f"Error with columns: {e}.")
        return

    df = df.dropna(subset=["peptide", "matched_15mer"])
    print("Mapping sequences and translation tables...")

    pep_data = defaultdict(lambda: defaultdict(set))  # pep -> 15mer -> set(tables)
    pep_to_tables = defaultdict(set)                  # pep -> set(tables)

    for pep, mer15, t_table in zip(df["peptide"], df["matched_15mer"], df["transl_table"]):
        clean_pep = str(pep).strip().upper()
        clean_15mer = str(mer15).strip().upper()

        if pd.notna(t_table):
            clean_t_table = str(t_table).strip().replace(".0", "")
        else:
            clean_t_table = "Unknown"

        pep_data[clean_pep][clean_15mer].add(clean_t_table)
        pep_to_tables[clean_pep].add(clean_t_table)

    peptides = set(pep_to_tables.keys())

    # --- Tracking Variables for Standard Pairs ---
    total_pep_pairs = 0
    std_pairs_with_15mer_match = 0
    std_pairs_sharing_table = 0
    std_shared_table_examples = []

    # --- Tracking Variables for Palindromes ---
    total_palindromes = 0
    pal_pep_sharing_table = 0
    palindrome_table_distribution = Counter()

    pal_with_same_15mer = 0
    pal_same_15mer_sharing_table = 0

    pal_with_diff_15mer_pairs = 0
    pal_diff_15mer_sharing_table = 0

    pal_same_15mer_examples = []

    # ============================================================
    # collect detailed rows for CSV export
    # ============================================================
    std_pairs_rows = []
    palindrome_rows = []
    palindrome_tables_rows = []
    pal_same15_rows = []
    pal_diff15_rows = []

    reverse_pair_table_distribution = Counter()

    print("Analyzing sequences...")
    for pep in peptides:
        rev_pep = pep[::-1]

        if rev_pep in peptides:

            # ==========================================
            # SCENARIO 1: PALINDROMIC PENTAPEPTIDES
            # ==========================================
            if pep == rev_pep:
                total_palindromes += 1

                # table-sharing across distinct 15-mers for the same palindromic peptide
                mers = list(pep_data[pep].items())  # [(15mer, set(tables)), ...]
                shared_tables_for_palindrome = set()
                palindrome_has_shared_table = False

                for i in range(len(mers)):
                    _, tset1 = mers[i]
                    for j in range(i + 1, len(mers)):
                        _, tset2 = mers[j]
                        shared = tset1.intersection(tset2)
                        if shared:
                            palindrome_has_shared_table = True
                            shared_tables_for_palindrome.update(shared)

                if palindrome_has_shared_table:
                    pal_pep_sharing_table += 1
                    for t in shared_tables_for_palindrome:
                        palindrome_table_distribution[t] += 1

                palindrome_rows.append({
                    "pep": pep,
                    "num_15mers": len(pep_data[pep]),
                    "tables_union": ",".join(sorted(pep_to_tables[pep])),
                    "has_shared_table_across_15mers": palindrome_has_shared_table,
                    "shared_tables_across_15mers": ",".join(sorted(shared_tables_for_palindrome)) if shared_tables_for_palindrome else ""
                })

                # 15-mer level analysis (original logic)
                has_same_15mer = False
                has_diff_15mer_pair = False
                diff_15mer_shared_table = False
                same_15mer_shared_table = False

                for m1, tables1 in pep_data[pep].items():
                    rev_m1 = m1[::-1]

                    if rev_m1 in pep_data[pep]:
                        # Case A: 15-mer is identical (palindromic 15-mer)
                        if m1 == rev_m1:
                            has_same_15mer = True

                            # logically equivalent to bool(tables1), kept as-is
                            if tables1.intersection(tables1):
                                same_15mer_shared_table = True

                            if len(pal_same_15mer_examples) < 10:
                                pal_same_15mer_examples.append({"pep": pep, "15mer": m1, "tables": tables1})

                            pal_same15_rows.append({
                                "pep": pep,
                                "15mer": m1,
                                "tables": ",".join(sorted(tables1))
                            })

                        # Case B: 15-mers are different but form a reverse pair (string reverse)
                        elif m1 < rev_m1:
                            has_diff_15mer_pair = True
                            shared = tables1.intersection(pep_data[pep][rev_m1])

                            if shared:
                                diff_15mer_shared_table = True
                                pal_diff15_rows.append({
                                    "pep": pep,
                                    "15mer_1": m1,
                                    "15mer_2": rev_m1,
                                    "shared_tables": ",".join(sorted(shared))
                                })

                if has_same_15mer:
                    pal_with_same_15mer += 1
                if same_15mer_shared_table:
                    pal_same_15mer_sharing_table += 1

                if has_diff_15mer_pair:
                    pal_with_diff_15mer_pairs += 1
                if diff_15mer_shared_table:
                    pal_diff_15mer_sharing_table += 1

            # ==========================================
            # SCENARIO 2: STANDARD REVERSED PAIRS
            # ==========================================
            elif pep < rev_pep:
                total_pep_pairs += 1

                has_15mer_match = False
                for m1 in pep_data[pep].keys():
                    if m1[::-1] in pep_data[rev_pep]:
                        has_15mer_match = True
                        break

                if has_15mer_match:
                    std_pairs_with_15mer_match += 1

                tables1 = pep_to_tables[pep]
                tables2 = pep_to_tables[rev_pep]
                shared_tables = tables1.intersection(tables2)

                if shared_tables:
                    std_pairs_sharing_table += 1
                    if len(std_shared_table_examples) < 10:
                        std_shared_table_examples.append({"pair": f"{pep} <-> {rev_pep}", "tables": shared_tables})

                    for t in shared_tables:
                        reverse_pair_table_distribution[t] += 1

                std_pairs_rows.append({
                    "pep": pep,
                    "rev_pep": rev_pep,
                    "pep_tables": ",".join(sorted(tables1)),
                    "rev_pep_tables": ",".join(sorted(tables2)),
                    "shared_tables": ",".join(sorted(shared_tables)) if shared_tables else "",
                    "has_shared_table": bool(shared_tables),
                    "has_reversed_15mer_match": has_15mer_match,
                    "num_shared_tables": len(shared_tables),
                })

    # ==========================================
    # PRINTING RESULTS
    # ==========================================
    print("\n" + "=" * 70)
    print("1. STANDARD REVERSED PENTAPEPTIDE PAIRS")
    print("=" * 70)
    print(f"Total standard reversed pentapeptide pairs: {total_pep_pairs}")
    print(f"-> Out of total, pairs with matching reversed 15-mers: {std_pairs_with_15mer_match}")

    std_pct = (std_pairs_sharing_table / total_pep_pairs * 100) if total_pep_pairs > 0 else 0
    print(f"-> Reversed pair pentapeptides sharing the same codon table: {std_pairs_sharing_table} ({std_pct:.2f}%)")

    print("\nSample of pairs with the same codon tables:")
    if std_shared_table_examples:
        for ex in std_shared_table_examples:
            print(f"  {ex['pair']} (Tables: {', '.join(ex['tables'])})")
    else:
        print("  None found.")

    print("\n" + "=" * 70)
    print("2. PALINDROMIC PENTAPEPTIDES")
    print("=" * 70)
    print(f"Total palindromic pentapeptides: {total_palindromes}")

    pal_pep_pct = (pal_pep_sharing_table / total_palindromes * 100) if total_palindromes > 0 else 0
    print(
        "-> Palindromic pentapeptides with ≥2 distinct 15-mers sharing a codon table: "
        f"{pal_pep_sharing_table} ({pal_pep_pct:.2f}%)"
    )

    print(f"-> Out of total, palindromes with EXACT SAME 15-mers (Palindromic 15-mers): {pal_with_same_15mer}")

    pal_15mer_pct = (pal_same_15mer_sharing_table / pal_with_same_15mer * 100) if pal_with_same_15mer > 0 else 0
    print(f"   -> Do these exact same 15-mers share a codon table?: {pal_same_15mer_sharing_table} ({pal_15mer_pct:.2f}%)")

    print(f"-> Out of total, palindromes with DIFFERENT 15-mers forming reverse pairs: {pal_with_diff_15mer_pairs}")

    pal_diff_pct = (pal_diff_15mer_sharing_table / pal_with_diff_15mer_pairs * 100) if pal_with_diff_15mer_pairs > 0 else 0
    print(
        f"   -> Reversed 15-mer pairs (within palindromes) sharing the same codon table: "
        f"{pal_diff_15mer_sharing_table} ({pal_diff_pct:.2f}%)"
    )

    print("\n" + "=" * 70)
    print("3. PALINDROME CODON TABLE DISTRIBUTION")
    print("=" * 70)
    print("Which specific codon tables do these palindromes (that share tables across 15-mers) live in?")
    if palindrome_table_distribution:
        for table, count in palindrome_table_distribution.most_common():
            print(f"  Table {table}: Contains {count} palindromes")
    else:
        print("  No distribution data found.")

    print("\n" + "-" * 70)
    print("Sample of palindromes with EXACT SAME 15-mers:")
    if pal_same_15mer_examples:
        for ex in pal_same_15mer_examples:
            print(f"  Peptide: {ex['pep']} | 15-mer: {ex['15mer']} (Tables: {', '.join(ex['tables'])})")
    else:
        print("  None found.")

    # Ensure output directory exists
    os.makedirs(out_dir, exist_ok=True)

    # ==========================================
    # SAVE RESULTS TO CSV
    # ==========================================
    palindrome_tables_rows = [{"transl_table": t, "palindrome_count": c}
                             for t, c in palindrome_table_distribution.most_common()]

    reverse_pair_tables_rows = [{"transl_table": t, "reverse_pair_count": c}
                               for t, c in reverse_pair_table_distribution.most_common()]

    std_pairs_csv = f"{out_dir}/standard_reversed_pairs.csv"
    palindromes_csv = f"{out_dir}/palindromic_peptides_summary.csv"
    pal_tables_csv = f"{out_dir}/palindrome_table_distribution.csv"
    pal_same15_csv = f"{out_dir}/palindromic_15mers.csv"
    pal_diff15_csv = f"{out_dir}/palindrome_reversed_15mer_pairs.csv"
    rev_tables_csv = f"{out_dir}/reverse_pair_table_distribution.csv"

    df_std = pd.DataFrame(std_pairs_rows)
    df_pal = pd.DataFrame(palindrome_rows)
    df_pal_tables = pd.DataFrame(palindrome_tables_rows)
    df_pal_same15 = pd.DataFrame(pal_same15_rows)
    df_pal_diff15 = pd.DataFrame(pal_diff15_rows)
    df_rev_tables = pd.DataFrame(reverse_pair_tables_rows)

    df_std.to_csv(std_pairs_csv, index=False)
    df_pal.to_csv(palindromes_csv, index=False)
    df_pal_tables.to_csv(pal_tables_csv, index=False)
    df_pal_same15.to_csv(pal_same15_csv, index=False)
    df_pal_diff15.to_csv(pal_diff15_csv, index=False)
    df_rev_tables.to_csv(rev_tables_csv, index=False)

    print("\nSaved CSV outputs:", flush=True)
    print(f"  - {std_pairs_csv}")
    print(f"  - {palindromes_csv}")
    print(f"  - {pal_tables_csv}")
    print(f"  - {rev_tables_csv}")
    print(f"  - {pal_same15_csv}")
    print(f"  - {pal_diff15_csv}")

    # ============================================================
    # PLOTS (improved styling, no "box", NO GRIDLINES)
    # ============================================================
    plt.rcParams.update({
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "font.size": 12,
        "axes.titlesize": 18,
        "axes.labelsize": 14,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
    })

    def style_ax(ax):
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["left"].set_linewidth(1.0)
        ax.spines["bottom"].set_linewidth(1.0)
        ax.grid(False)  # NO GRIDLINES
        ax.tick_params(axis="both", which="major", length=5, width=1)

    def save_fig(fig, path):
        fig.tight_layout()
        fig.savefig(path, bbox_inches="tight", facecolor="white")
        plt.close(fig)

    BLUE = "#1f77b4"
    ORANGE = "#ff7f0e"
    GREEN = "#2ca02c"
    RED = "#d62728"

    # Plot 1: Reverse pairs share-table proportion
    share_yes = std_pairs_sharing_table
    share_no = total_pep_pairs - std_pairs_sharing_table

    fig1, ax1 = plt.subplots(figsize=(7.2, 4.6))
    ax1.bar(["Share ≥1 table", "Share none"], [share_yes, share_no],
            color=[BLUE, ORANGE], edgecolor="none")
    ax1.set_title("Standard reversed peptide pairs: shared codon table?")
    ax1.set_ylabel("Number of peptide pairs")
    style_ax(ax1)
    plot1_path = f"{out_dir}/plot_reverse_pairs_share_table.png"
    save_fig(fig1, plot1_path)

    # Plot 2: Histogram of # shared tables per reverse pair
    fig2, ax2 = plt.subplots(figsize=(7.2, 4.6))
    if not df_std.empty and "num_shared_tables" in df_std.columns:
        vals = df_std["num_shared_tables"].fillna(0).astype(int)
        minv, maxv = int(vals.min()), int(vals.max())
        bins = [b - 0.5 for b in range(minv, maxv + 2)]
        ax2.hist(vals, bins=bins, color=BLUE, edgecolor="white", linewidth=1.0, alpha=0.9)
        ax2.set_xticks(list(range(minv, maxv + 1)))
        ax2.set_title("Standard reversed peptide pairs: # of shared codon tables")
        ax2.set_xlabel("Number of shared codon tables")
        ax2.set_ylabel("Number of peptide pairs")
        style_ax(ax2)
    else:
        ax2.text(0.5, 0.5, "No data to plot", ha="center", va="center")
        ax2.axis("off")
    plot2_path = f"{out_dir}/plot_reverse_pairs_num_shared_tables_hist.png"
    save_fig(fig2, plot2_path)

    # Plot 3: Palindromes share-table-across-15mers proportion
    pal_yes = pal_pep_sharing_table
    pal_no = total_palindromes - pal_pep_sharing_table

    fig3, ax3 = plt.subplots(figsize=(7.2, 4.6))
    ax3.bar(["Has shared table\n(across ≥2 15-mers)", "No shared table\n(across 15-mers)"],
            [pal_yes, pal_no],
            color=[GREEN, RED], edgecolor="none")
    ax3.set_title("Palindromic pentapeptides: shared codon table across 15-mers?")
    ax3.set_ylabel("Number of palindromic peptides")
    style_ax(ax3)
    plot3_path = f"{out_dir}/plot_palindromes_share_table_across_15mers.png"
    save_fig(fig3, plot3_path)

    # Plot 4: Palindrome codon table distribution (line plot)
    fig4, ax4 = plt.subplots(figsize=(7.6, 4.6))
    if not df_pal_tables.empty:
        df_pal_tables_sorted = df_pal_tables.sort_values("palindrome_count", ascending=False)
        ax4.plot(
            df_pal_tables_sorted["transl_table"].astype(str),
            df_pal_tables_sorted["palindrome_count"],
            marker="o",
            linewidth=2.2,
            color=GREEN
        )
        ax4.set_title("Palindromic peptides: codon table distribution (shared tables)")
        ax4.set_xlabel("Translation table")
        ax4.set_ylabel("Count")
        style_ax(ax4)
    else:
        ax4.text(0.5, 0.5, "No distribution data to plot", ha="center", va="center")
        ax4.axis("off")
    plot4_path = f"{out_dir}/plot_palindrome_table_distribution.png"
    save_fig(fig4, plot4_path)

    # Plot 5: Histogram of # distinct 15-mers per palindromic peptide
    fig5, ax5 = plt.subplots(figsize=(7.2, 4.6))
    if not df_pal.empty and "num_15mers" in df_pal.columns:
        vals = df_pal["num_15mers"].fillna(0).astype(int)
        minv, maxv = int(vals.min()), int(vals.max())
        bins = [b - 0.5 for b in range(minv, maxv + 2)]
        ax5.hist(vals, bins=bins, color=GREEN, edgecolor="white", linewidth=1.0, alpha=0.9)
        ax5.set_xticks(list(range(minv, maxv + 1)))
        ax5.set_title("Palindromic pentapeptides: # of distinct mapped 15-mers")
        ax5.set_xlabel("Number of distinct 15-mers")
        ax5.set_ylabel("Number of palindromic peptides")
        style_ax(ax5)
    else:
        ax5.text(0.5, 0.5, "No data to plot", ha="center", va="center")
        ax5.axis("off")
    plot5_path = f"{out_dir}/plot_palindromes_num_15mers_hist.png"
    save_fig(fig5, plot5_path)

    # Plot 6: Reverse-pair codon table distribution (line plot)
    fig6, ax6 = plt.subplots(figsize=(7.6, 4.6))
    if not df_rev_tables.empty:
        df_rev_sorted = df_rev_tables.sort_values("reverse_pair_count", ascending=False)
        ax6.plot(
            df_rev_sorted["transl_table"].astype(str),
            df_rev_sorted["reverse_pair_count"],
            marker="o",
            linewidth=2.2,
            color=BLUE
        )
        ax6.set_title("Standard reversed peptide pairs: codon table distribution (shared tables)")
        ax6.set_xlabel("Translation table")
        ax6.set_ylabel("Count of reverse pairs")
        style_ax(ax6)
    else:
        ax6.text(0.5, 0.5, "No reverse-pair distribution data to plot", ha="center", va="center")
        ax6.axis("off")
    plot6_path = f"{out_dir}/plot_reverse_pair_table_distribution.png"
    save_fig(fig6, plot6_path)

    print("\nSaved improved plot images:", flush=True)
    print(f"  - {plot1_path}")
    print(f"  - {plot2_path}")
    print(f"  - {plot3_path}")
    print(f"  - {plot4_path}")
    print(f"  - {plot5_path}")
    print(f"  - {plot6_path}")


# --- Execution (Colab) ---
if __name__ == "__main__":
    file_name = r"/content/final_pep_frames_tranls.csv"
    analyze_complete_peptides(file_name, out_dir="/content/")

In [ ]:
import pandas as pd
from google.colab import files

# Upload the supplementary Excel file
uploaded = files.upload()

# Get uploaded filename
input_file = list(uploaded.keys())[0]
print("Uploaded file:", input_file)

# Read the two main sheets
s2 = pd.read_excel(input_file, sheet_name="S2. PHASE I Ribo-seq ORFs")
s3 = pd.read_excel(input_file, sheet_name="S3. Single-study Ribo-seq ORFs")

# Add source labels
s2["source_sheet"] = "S2_replicated_ORFs"
s3["source_sheet"] = "S3_single_study_ORFs"

# Combine both sheets
combined = pd.concat([s2, s3], ignore_index=True)

# Basic checks
print("S2 rows:", len(s2))
print("S3 rows:", len(s3))
print("Combined rows:", len(combined))
print("Unique ORFs:", combined["orf_name"].nunique())

# Save outputs
combined.to_excel("combined_7264_Riboseq_ORFs.xlsx", index=False)
combined.to_csv("combined_7264_Riboseq_ORFs.csv", index=False)

# Download files
files.download("combined_7264_Riboseq_ORFs.xlsx")
files.download("combined_7264_Riboseq_ORFs.csv")

In [ ]:
import pandas as pd
from google.colab import files
from tqdm.auto import tqdm

# Upload:
# 1. combined_7264_Riboseq_ORFs.csv
# 2. final_absent_pep_tranls.csv
uploaded = files.upload()

orf_file = "combined_7264_Riboseq_ORFs.csv"
pep_file = "final_absent_pep_tranls.csv"

print("Reading files...")
orf_df = pd.read_csv(orf_file)
pep_df = pd.read_csv(pep_file)

print("ORF dataframe:", orf_df.shape)
print("Peptide dataframe:", pep_df.shape)
print("Peptide columns:", pep_df.columns.tolist())

peptide_col = "pentapeptide"

# Clean peptide list
pep_df[peptide_col] = (
    pep_df[peptide_col]
    .astype(str)
    .str.strip()
    .str.upper()
)

query_peptides = sorted(set(pep_df[peptide_col].dropna()))

query_peptides = [
    p for p in query_peptides
    if len(p) == 5 and p.isalpha()
]

query_set = set(query_peptides)

print("Unique valid pentapeptides:", len(query_set))

# Clean ORF sequences
orf_df["orf_sequence"] = (
    orf_df["orf_sequence"]
    .astype(str)
    .str.strip()
    .str.upper()
)

matches = []

print("\nSearching pentapeptides in ncORF sequences...\n")

for idx, row in tqdm(
    orf_df.iterrows(),
    total=len(orf_df),
    desc="Scanning 7,264 ncORFs"
):
    seq = row["orf_sequence"]

    if not isinstance(seq, str) or seq == "" or seq == "NAN":
        continue

    seq_len = len(seq)

    for i in range(seq_len - 4):
        pentamer = seq[i:i+5]

        if pentamer in query_set:
            context_start = max(0, i - 10)
            context_end = min(seq_len, i + 5 + 10)

            matches.append({
                "matched_peptide": pentamer,
                "orf_name": row.get("orf_name", ""),
                "orf_biotype": row.get("orf_biotype", ""),
                "gene_name": row.get("gene_name", ""),
                "gene_id": row.get("gene_id", ""),
                "transcript": row.get("transcript", ""),
                "chrm": row.get("chrm", ""),
                "strand": row.get("strand", ""),
                "orf_length": row.get("orf_length", seq_len),
                "match_start_1based": i + 1,
                "match_end_1based": i + 5,
                "context_sequence": seq[context_start:context_end],
                "MS_peptides": row.get("MS_peptides", ""),
                "number_studies_MS": row.get("number_studies_MS", ""),
                "Riboseq_evidence": row.get("Riboseq_evidence", ""),
                "source_sheet": row.get("source_sheet", "")
            })

matches_df = pd.DataFrame(matches)

matched_peptides = set(matches_df["matched_peptide"]) if len(matches_df) > 0 else set()
unmatched_peptides = sorted(query_set - matched_peptides)

summary_df = pd.DataFrame([{
    "total_query_pentapeptides": len(query_set),
    "matched_unique_pentapeptides": len(matched_peptides),
    "unmatched_unique_pentapeptides": len(unmatched_peptides),
    "total_match_events": len(matches_df),
    "unique_ncORFs_with_matches": matches_df["orf_name"].nunique() if len(matches_df) > 0 else 0,
    "unique_genes_with_matches": matches_df["gene_name"].nunique() if len(matches_df) > 0 else 0
}])

unmatched_df = pd.DataFrame({
    "unmatched_peptide": unmatched_peptides
})

print("\nSUMMARY")
display(summary_df)

print("\nTop matches")
display(matches_df.head())

# Save outputs
matches_df.to_csv("final_absent_pentapeptides_matched_in_7264_ncORFs.csv", index=False)
unmatched_df.to_csv("final_absent_pentapeptides_not_found_in_7264_ncORFs.csv", index=False)
summary_df.to_csv("final_absent_pentapeptide_ncORF_search_summary.csv", index=False)

files.download("final_absent_pentapeptides_matched_in_7264_ncORFs.csv")
files.download("final_absent_pentapeptides_not_found_in_7264_ncORFs.csv")
files.download("final_absent_pentapeptide_ncORF_search_summary.csv")

print("DONE.")

In [ ]:
# ============================================================
# ADD 2026 PROTEIN-LEVEL EVIDENCE ANNOTATION
# ============================================================

# Upload the 2026 supplementary files:
# 1. 41586_2026_10459_MOESM13_ESM.xlsx  -> ORF-level protein evidence
# 2. 41586_2026_10459_MOESM14_ESM.xlsx  -> peptide-level MS evidence

print("\nUpload 2026 supplementary evidence files...")
uploaded_evidence = files.upload()

# File names
orf_evidence_file = "41586_2026_10459_MOESM13_ESM.xlsx"
peptide_evidence_file = "41586_2026_10459_MOESM14_ESM.xlsx"

# Read ORF-level evidence file
evidence_orf_df = pd.read_excel(
    orf_evidence_file,
    sheet_name="Table S3"
)

# Read peptide-level evidence file
evidence_peptide_df = pd.read_excel(
    peptide_evidence_file,
    sheet_name="Table S2"
)

# Clean key columns
matches_df["orf_name"] = matches_df["orf_name"].astype(str).str.strip()
evidence_orf_df["Ribo-Seq_ORF"] = evidence_orf_df["Ribo-Seq_ORF"].astype(str).str.strip()
evidence_peptide_df["Ribo-Seq_ORF"] = evidence_peptide_df["Ribo-Seq_ORF"].astype(str).str.strip()

# ============================================================
# MATCH 703 PEPTIDE-HARBORING ncORFs WITH 2026 PROTEIN EVIDENCE
# ============================================================

protein_level_matches = matches_df.merge(
    evidence_orf_df,
    left_on="orf_name",
    right_on="Ribo-Seq_ORF",
    how="inner",
    suffixes=("_matched", "_2026")
)

print("\nProtein-level evidence matches found:")
print("Total match rows:", len(protein_level_matches))
print("Unique matched peptides:", protein_level_matches["matched_peptide"].nunique())
print("Unique ncORFs with protein-level evidence:", protein_level_matches["orf_name"].nunique())

# ============================================================
# DEFINE HIGH-CONFIDENCE EVIDENCE
# Here, Tier 2A is treated as high-confidence non-HLA protein-level evidence
# ============================================================

protein_level_matches["Final.tier.for.level.of.evidence"] = (
    protein_level_matches["Final.tier.for.level.of.evidence"]
    .astype(str)
    .str.strip()
)

high_confidence_matches = protein_level_matches[
    protein_level_matches["Final.tier.for.level.of.evidence"].str.contains(
        "Tier 2A",
        case=False,
        na=False
    )
].copy()

print("\nHigh-confidence protein-level evidence matches:")
print("Total high-confidence rows:", len(high_confidence_matches))
print("Unique high-confidence peptides:", high_confidence_matches["matched_peptide"].nunique())
print("Unique high-confidence ncORFs:", high_confidence_matches["orf_name"].nunique())

# ============================================================
# ADD PEPTIDE-LEVEL MS DETAILS FROM TABLE S2
# ============================================================

protein_level_with_ms_details = protein_level_matches.merge(
    evidence_peptide_df,
    left_on="orf_name",
    right_on="Ribo-Seq_ORF",
    how="left",
    suffixes=("", "_peptideMS")
)

high_confidence_with_ms_details = high_confidence_matches.merge(
    evidence_peptide_df,
    left_on="orf_name",
    right_on="Ribo-Seq_ORF",
    how="left",
    suffixes=("", "_peptideMS")
)

# ============================================================
# SAVE OUTPUT FILES
# ============================================================

protein_level_matches.to_csv(
    "703_peptides_with_2026_protein_level_ORF_evidence.csv",
    index=False
)

high_confidence_matches.to_csv(
    "703_peptides_with_2026_high_confidence_Tier2A_evidence.csv",
    index=False
)

protein_level_with_ms_details.to_csv(
    "703_peptides_with_2026_protein_level_and_MS_peptide_details.csv",
    index=False
)

high_confidence_with_ms_details.to_csv(
    "703_peptides_with_2026_high_confidence_and_MS_peptide_details.csv",
    index=False
)

# ============================================================
# SUMMARY TABLE
# ============================================================

evidence_summary_df = pd.DataFrame([{
    "total_ncORF_matched_peptides_from_703_set": matches_df["matched_peptide"].nunique(),
    "total_ncORFs_with_703_peptide_matches": matches_df["orf_name"].nunique(),

    "unique_peptides_with_2026_protein_level_evidence":
        protein_level_matches["matched_peptide"].nunique(),

    "unique_ncORFs_with_2026_protein_level_evidence":
        protein_level_matches["orf_name"].nunique(),

    "unique_peptides_with_high_confidence_Tier2A_evidence":
        high_confidence_matches["matched_peptide"].nunique(),

    "unique_ncORFs_with_high_confidence_Tier2A_evidence":
        high_confidence_matches["orf_name"].nunique()
}])

display(evidence_summary_df)

evidence_summary_df.to_csv(
    "703_peptide_2026_protein_evidence_summary.csv",
    index=False
)

# Download outputs
files.download("703_peptides_with_2026_protein_level_ORF_evidence.csv")
files.download("703_peptides_with_2026_high_confidence_Tier2A_evidence.csv")
files.download("703_peptides_with_2026_protein_level_and_MS_peptide_details.csv")
files.download("703_peptides_with_2026_high_confidence_and_MS_peptide_details.csv")
files.download("703_peptide_2026_protein_evidence_summary.csv")

print("\n2026 protein-level evidence extraction complete.")